# 07 · Duplicate retrieval with sentence vectors

The improved method for task B. It is evaluated exactly like BM25 in 06: the same validation queries, the same
time-aware pools (04), the same ranking rule and metrics (`src/retrieval.py`). Only the score changes: the
**cosine similarity** of sentence vectors instead of shared word stems.

**Input** (in `data/processed/`): `petitions.parquet`, `gold_pairs.parquet`, `splits.parquet`, the vectors from
`colab_gpu.ipynb` (`embeddings/`), and BM25's ranks from 06 (`predictions/bm25_val_ranks.parquet`).

**Output:**

| File | Content |
|---|---|
| `data/processed/predictions/embedding_val_ranks.parquet` | per validation query: rank of the first correct petition for MiniLM, mpnet and the hybrid, open and stress pool |
| `reports/retrieval_val_results.csv` | Recall@1/5/10 and MRR of BM25, MiniLM, mpnet and the hybrid, open and stress pool |
| `reports/hybrid_alpha_val.csv` | the hybrid's validation scores for each α |

Steps: 1 load → 2 sentence vectors in plain words → 3 evaluate → 4 BM25 vs vectors → 5 hybrid → 6 where the
correct petition lands → 7 examples → 8 save. Runs on the laptop in about 2 minutes. The test split is not used.

In [1]:
import sys, time
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent                      # this notebook runs inside notebooks/
sys.path.insert(0, str(ROOT / "src"))
from preprocess import MODEL_INPUTS, leakage_check
from splits import CandidatePools
from retrieval import BM25Index, gold_rank, hybrid_scores, retrieval_metrics

DATA = ROOT / "data" / "processed"
MODELS = {"minilm": "all-MiniLM-L6-v2", "mpnet": "all-mpnet-base-v2"}
SEED = 42

## 1. Load

In [2]:
columns = ["id", "label", "action", "text", "decision_time", "opened_at", "closed_at"]
petitions = pd.read_parquet(DATA / "petitions.parquet", columns=columns)
petitions = petitions.merge(pd.read_parquet(DATA / "splits.parquet"), on="id", validate="one_to_one")
gold = pd.read_parquet(DATA / "gold_pairs.parquet")
pools = CandidatePools(petitions)
by_id = petitions.set_index("id")

val_queries = petitions[(petitions["split"] == "val") & petitions["id"].isin(gold["query_id"])]
gold_sets = gold.groupby("query_id")["target_id"].apply(set)
assert len(val_queries) == 1_155
leakage_check(val_queries[MODEL_INPUTS])        # the vectors were made from the text only (colab_gpu)
leakage_check(by_id.loc[pools.ids, "text"])
print(f"{len(val_queries):,} validation queries, {len(pools.ids):,} published petitions")

1,155 validation queries, 40,381 published petitions


## 2. Sentence vectors in plain words

A sentence model reads a whole petition and turns it into a list of numbers (a **vector**: 384 numbers for MiniLM,
768 for mpnet). The models were trained on millions of sentence pairs so that texts **with the same meaning get
vectors pointing in the same direction**, even when they use different words ("Dissolution of Parliament" /
"Call a general election").

- The vectors were computed once on Colab (`colab_gpu.ipynb`) and have length 1, so the **cosine similarity** of
  two petitions is simply their dot product: 1 = same direction, around 0.2 = unrelated petitions.
- For a query, every petition in its pool gets the cosine with the query as its score; the rest is as in 06.
- MiniLM is small and fast (6 layers); mpnet is larger (12 layers) and usually more accurate.

The vectors are read from disk only for the petitions needed here (the published ones and the queries).

In [3]:
def load_vectors(name, wanted_ids):
    """Vectors of the petitions in wanted_ids, in that order, read without loading the whole file."""
    ids = np.load(DATA / "embeddings" / f"{name}_ids.npy")
    vectors = np.load(DATA / "embeddings" / f"{name}_vectors.npy", mmap_mode="r")
    row = pd.Series(np.arange(len(ids)), index=ids)
    return np.asarray(vectors[row.loc[wanted_ids].to_numpy()])

def evaluate(pool_vectors, query_vectors, stress=False):
    """Rank of the first correct petition for every validation query, and milliseconds per query."""
    ranks, start = [], time.time()
    for query_vector, query_id, t in zip(query_vectors, val_queries["id"], val_queries["decision_time"]):
        mask = pools.mask(t, query_id, stress)
        scores = (pool_vectors @ query_vector)[mask]       # cosine with every published petition, then keep the pool
        ranks.append(gold_rank(scores, pools.ids[mask], gold_sets[query_id]))
    return np.array(ranks, dtype=float), (time.time() - start) / len(val_queries) * 1000

## 3. Evaluate MiniLM and mpnet

In [4]:
rows, ranks = [], {}
for name in MODELS:
    pool_vectors = load_vectors(name, pools.ids)               # rows in pools.ids order, like every pool mask
    query_vectors = load_vectors(name, val_queries["id"])
    for pool in ["open", "stress"]:
        ranks[name, pool], ms = evaluate(pool_vectors, query_vectors, stress=(pool == "stress"))
        rows.append({"method": name, "pool": pool, **retrieval_metrics(ranks[name, pool]), "ms_per_query": round(ms, 1)})
    print(f"{name}: {pool_vectors.shape[1]} numbers per petition")
del pool_vectors, query_vectors

bm25 = pd.read_parquet(DATA / "predictions" / "bm25_val_ranks.parquet").set_index("query_id").loc[val_queries["id"]]
bm25_table = pd.read_csv(ROOT / "reports" / "bm25_val_results.csv")
BEST_K1 = float(bm25_table.query("pool == 'open'").sort_values("mrr").iloc[-1]["k1"])   # chosen in 06
bm25_ms = bm25_table.query("k1 == @BEST_K1").set_index("pool")["ms_per_query"]
for pool in ["open", "stress"]:
    ranks["bm25", pool] = bm25[f"rank_{pool}"].to_numpy()
    rows.append({"method": "bm25", "pool": pool, **retrieval_metrics(ranks["bm25", pool]), "ms_per_query": bm25_ms[pool]})

results = pd.DataFrame(rows).set_index(["pool", "method"]).sort_index(ascending=[True, True])
results.round(4)

minilm: 384 numbers per petition


mpnet: 768 numbers per petition


recall@1  recall@5  recall@10     mrr  ms_per_query
pool   method                                                     
open   bm25      0.5238    0.8043     0.8719  0.6488           3.9
       minilm    0.6727    0.9065     0.9584  0.7749           1.8
       mpnet     0.6502    0.9039     0.9463  0.7629           3.0
stress bm25      0.1680    0.3905     0.5177  0.2813          25.4
       minilm    0.2225    0.4900     0.6390  0.3514           1.8
       mpnet     0.1732    0.4589     0.6130  0.3100           2.9

## 4. BM25 vs vectors

The best vector model by open-pool MRR, compared query by query with BM25: for how many queries does each
method put a correct petition in the top 5 (what a moderator would see)?

In [5]:
BEST = results.loc["open"].drop(index="bm25")["mrr"].idxmax()
print("best vector model:", BEST)

hit_bm25 = ranks["bm25", "open"] <= 5
hit_best = ranks[BEST, "open"] <= 5
pd.crosstab(pd.Series(hit_bm25, name="BM25 top 5"), pd.Series(hit_best, name=f"{BEST} top 5"), margins=True)

best vector model: minilm


minilm top 5,False,True,All
BM25 top 5,,,
False,78,148,226
True,30,899,929
All,108,1047,1155


## 5. Hybrid: BM25 + vectors

The table above shows that the two methods do not fail on the same queries: BM25 sometimes finds a petition with
the same key words that the vectors rank lower. The hybrid (optional in the plan) mixes both scores:

score = α × BM25 / (best BM25 score in the pool) + (1 − α) × cosine

Dividing by the pool's best BM25 score puts BM25 on a 0–1 scale, like the cosine. α = 0 is the vectors alone and
α = 1 is BM25 alone; both ends are checked against sections 3 and 06. α is chosen on validation by MRR, so the
hybrid's validation score is slightly optimistic; α is frozen and judged on the test split in 09.

In [6]:
index = BM25Index(by_id.loc[pools.ids, "text"].tolist())      # the same BM25 as in 06
pool_vectors = load_vectors(BEST, pools.ids)
query_vectors = load_vectors(BEST, val_queries["id"])
ALPHAS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 1.0]

def evaluate_hybrid(alphas, stress=False):
    """Ranks for each alpha, and milliseconds per query."""
    hybrid_ranks, start = {a: [] for a in alphas}, time.time()
    for query_vector, query_id, text, t in zip(query_vectors, val_queries["id"], val_queries["text"], val_queries["decision_time"]):
        mask = pools.mask(t, query_id, stress)
        cosine = (pool_vectors @ query_vector)[mask]
        bm25_scores = index.scores(text, mask, k1=BEST_K1)
        for a in alphas:
            hybrid_ranks[a].append(gold_rank(hybrid_scores(bm25_scores, cosine, a), pools.ids[mask], gold_sets[query_id]))
    return {a: np.array(r, dtype=float) for a, r in hybrid_ranks.items()}, (time.time() - start) / len(val_queries) * 1000

grid, _ = evaluate_hybrid(ALPHAS)
assert (grid[0.0] == ranks[BEST, "open"]).all() and (grid[1.0] == ranks["bm25", "open"]).all()
alpha_table = pd.DataFrame([{"alpha": a, **retrieval_metrics(r)} for a, r in grid.items()])
BEST_ALPHA = float(alpha_table.loc[alpha_table["mrr"].idxmax(), "alpha"])
print("best alpha:", BEST_ALPHA)
alpha_table.round(4)

best alpha: 0.1


,alpha,recall@1,recall@5,recall@10,mrr
0,0.0,0.6727,0.9065,0.9584,0.7749
1,0.1,0.6857,0.9221,0.9662,0.7867
2,0.2,0.6797,0.9247,0.9619,0.7839
3,0.3,0.6615,0.9117,0.9610,0.7704
4,0.4,0.6407,0.8987,0.9532,0.7535
5,0.5,0.6017,0.8814,0.9359,0.7242
6,1.0,0.5238,0.8043,0.8719,0.6488


In [7]:
for pool in ["open", "stress"]:
    hybrid, ms = evaluate_hybrid([BEST_ALPHA], stress=(pool == "stress"))
    ranks["hybrid", pool] = hybrid[BEST_ALPHA]
    rows.append({"method": "hybrid", "pool": pool, **retrieval_metrics(ranks["hybrid", pool]), "ms_per_query": round(ms, 1)})
assert (ranks["hybrid", "open"] == grid[BEST_ALPHA]).all()
results = pd.DataFrame(rows).set_index(["pool", "method"]).sort_index()
results.round(4)

recall@1  recall@5  recall@10     mrr  ms_per_query
pool   method                                                     
open   bm25      0.5238    0.8043     0.8719  0.6488           3.9
       hybrid    0.6857    0.9221     0.9662  0.7867           6.7
       minilm    0.6727    0.9065     0.9584  0.7749           1.8
       mpnet     0.6502    0.9039     0.9463  0.7629           3.0
stress bm25      0.1680    0.3905     0.5177  0.2813          25.4
       hybrid    0.2459    0.5195     0.6589  0.3770          27.8
       minilm    0.2225    0.4900     0.6390  0.3514           1.8
       mpnet     0.1732    0.4589     0.6130  0.3100           2.9

## 6. Where the correct petition lands (open pool)

In [8]:
bins = [0, 1, 5, 10, 50, np.inf]
labels = ["1", "2-5", "6-10", "11-50", "> 50"]
landing = {}
for method in ["bm25", "minilm", "mpnet", "hybrid"]:
    r = ranks[method, "open"]
    counts = pd.cut(r[np.isfinite(r)], bins=bins, labels=labels).value_counts().reindex(labels)
    counts["not in the pool"] = int(np.isinf(r).sum())
    landing[method] = counts
pd.DataFrame(landing)

,bm25,minilm,mpnet,hybrid
1,605,777,751,792
2-5,324,270,293,273
6-10,78,60,49,51
11-50,105,35,54,28
> 50,41,11,6,9
not in the pool,2,2,2,2


## 7. Examples

In [9]:
# The audit's paraphrase pair (a train query; BM25 put the correct petition at rank 173)
query = by_id.loc[150557]
mask = pools.mask(query["decision_time"], 150557)
pool_ids = pools.ids[mask]
for name in MODELS:
    scores = load_vectors(name, pool_ids) @ load_vectors(name, [150557])[0]
    top = pool_ids[np.argsort(-scores)[:3]]
    print(f"{name}: #121973 at rank {gold_rank(scores, pool_ids, gold_sets[150557]):.0f} of {mask.sum():,}; top 3:",
          [by_id.loc[pid, 'action'] for pid in top])

minilm: #121973 at rank 1 of 3,187; top 3: ['Stop TV licensing!', 'Stop The TV licence by Privatising the BBC.', 'End the compulsory licence fee for those who do not use BBC services.']
mpnet: #121973 at rank 1 of 3,187; top 3: ['Stop TV licensing!', 'Stop The TV licence by Privatising the BBC.', 'Force the BBC to scrap the licence fee in favour of funding from advertisements.']


In [10]:
def side_by_side(selected, n=4):
    """Query title, correct petition title and both ranks for n random queries."""
    picked = val_queries[selected].sample(min(n, selected.sum()), random_state=SEED)
    position = pd.Series(np.arange(len(val_queries)), index=val_queries["id"]).loc[picked["id"]]
    return pd.DataFrame({"query": picked["action"].to_numpy(),
                         "correct petition": [by_id.loc[min(gold_sets[q]), "action"] for q in picked["id"]],
                         "bm25 rank": ranks["bm25", "open"][position],
                         f"{BEST} rank": ranks[BEST, "open"][position]})

print(f"Vectors find it, BM25 does not ({BEST} top 5, BM25 below 10):")
side_by_side((ranks[BEST, "open"] <= 5) & (ranks["bm25", "open"] > 10))

Vectors find it, BM25 does not (minilm top 5, BM25 below 10):


,query,correct petition,bm25 rank,minilm rank
0,Strengthening Democracy Through Participation.,Introduce compulsory voting,14.0,3.0
1,The 5 years ILR route for existing skilled wor...,Keep 5-year ILR for Skilled Workers & dependen...,21.0,2.0
2,ALLOW UK CITIZENS TO HAVE A REFERENDUM ON DIGI...,Hold a referendum on introducing mandatory ide...,18.0,5.0
3,Stop the potential cancellation of the next Un...,Remove power to cancel local government elections,12.0,1.0


In [11]:
print(f"BM25 finds it, the vectors do not (BM25 top 5, {BEST} below 10):")
side_by_side((ranks["bm25", "open"] <= 5) & (ranks[BEST, "open"] > 10))

BM25 finds it, the vectors do not (BM25 top 5, minilm below 10):


,query,correct petition,bm25 rank,minilm rank
0,Stop the Government taking more of our hard ea...,No more tax rises in this Parliament,3.0,16.0
1,Protect our green spaces and allotments stop t...,Ban councils from selling allotments,1.0,24.0
2,Proposal for Flexible School Holiday Allowance,Allow parents to take their children out of sc...,3.0,16.0
3,Early General Election,Call an immediate general election,1.0,15.0


## 8. Save

In [12]:
pd.DataFrame({"query_id": val_queries["id"].to_numpy(),
              **{f"{name}_{pool}": ranks[name, pool] for name in [*MODELS, "hybrid"] for pool in ["open", "stress"]}}
             ).to_parquet(DATA / "predictions" / "embedding_val_ranks.parquet", index=False)
results.round(4).reset_index().to_csv(ROOT / "reports" / "retrieval_val_results.csv", index=False)
alpha_table.round(4).to_csv(ROOT / "reports" / "hybrid_alpha_val.csv", index=False)
print("saved; best vector model:", BEST, "| best alpha:", BEST_ALPHA)

saved; best vector model: minilm | best alpha: 0.1


## Summary

- Same 1,155 validation queries, pools, ranking rule and metrics as BM25 in 06; only the score changes.
- **The vectors beat BM25 clearly.** Open pool: MiniLM Recall@5 **0.907** and MRR **0.775**, mpnet 0.904 / 0.763,
  BM25 0.804 / 0.649. The small MiniLM is a little better than mpnet and faster (about 2 ms per query), so
  **MiniLM** is the vector model used from here on (similarity features in 08, the pipeline).
- **They catch paraphrases:** the audit pair "Ban the t.v licence" → "Stop TV licensing!" moves from rank 173
  (BM25) to rank 1. 148 queries reach the top 5 only with MiniLM, but 30 only with BM25
  (e.g. "Early General Election" → "Call an immediate general election": BM25 rank 1, MiniLM rank 15).
- **Hybrid** (α = 0.1, that is 10% BM25 and 90% MiniLM, chosen by MRR): Recall@1 0.686, Recall@5 **0.922**,
  Recall@10 0.966, MRR **0.787**, the best method on validation in both pools. Both ends of the α scale reproduce
  MiniLM and BM25 exactly. The score is slightly optimistic because α was chosen on these queries; α is frozen
  and judged on test in 09.
- **Stress pool:** every method drops a lot (hybrid Recall@5 0.520, BM25 0.391), so the time-aware pool matters
  more than the choice of method.
- In the open pool the hybrid ranks a correct petition first for 792 of the 1,155 queries (69%); 2 queries have
  no correct petition in their pool.

Next (week C): 08 uses MiniLM for the similarity features (top-1 cosine, mean of the top 5, top-1 − top-2 gap,
pool size) and combines them with the classifiers.
